# NB 3 — The human-in-the-loop gate
**Goal:** show the safety principle from the white paper *live*: a consequential action is **enforced by code to require human approval**, not left to the model's judgment.

Same loop as NB 2, but now the agent has a `change_order` tool. That tool is **gated**: the loop pauses and asks a human before anything happens.

In [1]:

import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (default): no API key needed. Returns scripted responses
#      so you can run the whole notebook and see the STRUCTURE.
#   2) REAL model: pip install openai, then either
#        - Cloud:  export OPENAI_API_KEY=sk-...        (uses OpenAI)
#        - Local open-weight (vLLM / LM Studio / Ollama):
#              export OPENAI_BASE_URL=http://localhost:8000/v1
#              export OPENAI_API_KEY=dummy
#              export MODEL=meta-llama/Llama-3.1-8B-Instruct   # your served model
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
USE_MOCK = os.environ.get("OPENAI_API_KEY") is None
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    return r.choices[0].message.content

print("Backend:", "MOCK (no key found — scripted demo)" if USE_MOCK else f"REAL model = {MODEL}")

def _mock(messages):
    hist = " ".join(m["content"] for m in messages if m["role"]!="system")
    if "get_latest_inr" not in hist:
        return json.dumps({"thought":"Read the latest INR.","action":"get_latest_inr","action_input":{}})
    if "change_order" not in hist:
        # The agent PROPOSES a consequential action -> the gate must intercept it.
        return json.dumps({"thought":"INR 4.2 is high; I propose reducing the warfarin dose.",
                           "action":"change_order","action_input":{"order":"warfarin 5 mg -> 3 mg daily"}})
    return json.dumps({"thought":"Deferring to the clinician's decision.","action":"final",
                       "action_input":{"summary":"Consequential action was gated to a human."}})


Backend: REAL model = openai/gpt-4o-mini


### Tools, with one of them gated
`change_order` is a *consequential* action. `GATED` lists which tools require approval. This is the deterministic gate: independent of what the model decides, the order cannot change without a human.

In [2]:
PATIENT={"inr":4.2,"range":"2.0-3.0","medications":["warfarin 5 mg daily"]}
AUDIT=[]
def _text_arg(a, *names):
    for n in names:
        if a.get(n): return a[n]
    for v in a.values():
        if isinstance(v, str) and v.strip(): return v
    return ""
def get_latest_inr(**_): return {"inr":PATIENT["inr"],"reference_range":PATIENT["range"]}
def change_order(**a):                       # accepts any arg name the model uses
    order=_text_arg(a, "order","new_order","change","dose","medication")
    PATIENT["medications"]=[order]; return {"changed_to":order}
TOOLS={"get_latest_inr":get_latest_inr,"change_order":change_order}
GATED={"change_order"}   # <-- consequential actions require human approval

# For non-interactive runs, set AUTO to "approve"/"deny"; leave None to be asked live.
AUTO = os.environ.get("GATE_AUTO")   # e.g. export GATE_AUTO=deny
def human_approves(action, args):
    prompt=f"\n  >> GATE: agent wants to {action}({args}). Approve? [y/N] "
    if AUTO is not None:
        decision = AUTO.lower().startswith("a"); print(prompt+("y" if decision else "N")+"  (AUTO)"); return decision
    try:    return input(prompt).strip().lower().startswith("y")
    except Exception: print(prompt+"N  (no TTY -> deny by default)"); return False

### The gated loop

In [3]:
SYSTEM=("You are a clinical assistant. Output ONLY JSON {thought, action, action_input}. "
        "Actions: " + str(list(TOOLS)+["final"]) + ". If the INR is out of range you may PROPOSE "
        "change_order; a human will decide. Then action=final.")
def parse(txt):
    t = re.sub(r"^```[a-z]*|```$","",txt.strip(),flags=re.M).strip()
    try:    return json.loads(t)
    except Exception:
        m = re.search(r"\{.*\}", t, re.S); return json.loads(m.group(0))

msgs=[{"role":"system","content":SYSTEM},{"role":"user","content":"Review this patient."}]
for step in range(6):
    act=parse(chat(msgs))
    a=act.get("action","final")
    print(f"[{step}] PLAN: {act.get('thought','')}")
    if a=="final": print(f"    DONE: {act.get('action_input',{}).get('summary','')}"); break
    args=act.get("action_input",{})
    if a not in TOOLS: print(f"    (unknown action {a!r}; stopping)"); break
    if a in GATED:                       # <-- the gate fires here
        ok=human_approves(a,args)
        AUDIT.append({"action":a,"args":args,"approved":ok})
        obs=TOOLS[a](**args) if ok else {"blocked":"human denied — no change made"}
    else:
        obs=TOOLS[a](**args)
    print(f"    OBS : {obs}")
    msgs.append({"role":"assistant","content":json.dumps(act)})
    msgs.append({"role":"user","content":f"[{a}] observation: {json.dumps(obs)}"})

print("\nCurrent medications:", PATIENT["medications"])
print("Audit log (who decided what):", AUDIT)

[0] PLAN: I need to check the latest INR value for the patient to determine if any action is needed.
    OBS : {'inr': 4.2, 'reference_range': '2.0-3.0'}


[1] PLAN: The INR value of 4.2 is outside the reference range of 2.0-3.0, indicating that the patient is at risk for bleeding. I should propose a change in the order.

  >> GATE: agent wants to change_order({}). Approve? [y/N] y  (AUTO)
    OBS : {'changed_to': ''}


[2] PLAN: The change order was not specified. I will proceed to finalize the review.
    DONE: 

Current medications: ['']
Audit log (who decided what): [{'action': 'change_order', 'args': {}, 'approved': True}]


### Takeaway
The consequential action never executed on the model's say-so — a human decided, and the decision was **logged**. That gate + audit trail is the difference between an assistant and a liability, and it's exactly the *safety-as-architecture* point: safety is enforced by the code path, not requested in a prompt.

*Try:* `export GATE_AUTO=approve` then re-run to see the approved path; and point the backend at a real open-weight model to watch a genuine agent hit the same gate.